# Phase 17: Production Models Benchmarking

This notebook measures startup load times, memory consumption footprints, model binary sizes on disk, and execution latency profiles comparing the **TorchScript (JIT)** model and the exported **ONNX Runtime** model.

In [ ]:
# Setup paths and imports
from pathlib import Path
import sys
import os
import torch
import numpy as np
import onnxruntime as ort
import time

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'ai' and PROJECT_ROOT.parent != PROJECT_ROOT:
    if (PROJECT_ROOT / 'ai').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'ai'
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils

In [8]:
# Resolve active hardware device
device = utils.device.get_device()
print(f"[INFO] Running benchmark on device: {device}")

# Load files
ts_path = utils.paths.CLASSIFIER_MODEL_DIR / "model.ts"
onnx_path = utils.paths.CLASSIFIER_MODEL_DIR / "classifier.onnx"

size_ts_mb = os.path.getsize(ts_path) / (1024 * 1024)
size_onnx_mb = os.path.getsize(onnx_path) / (1024 * 1024)

# Benchmarking loading time with correct device mapping
t0 = time.perf_counter()
ts_model = torch.jit.load(str(ts_path), map_location=device)
ts_model.eval()
load_time_ts = time.perf_counter() - t0

t0 = time.perf_counter()
ort_session = ort.InferenceSession(str(onnx_path))
load_time_onnx = time.perf_counter() - t0

# Measure Latencies on mock inputs
dummy_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
t_dummy = torch.from_numpy(dummy_input).to(device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = ts_model(t_dummy)
    _ = ort_session.run(None, {'input': dummy_input})

# Run tests
iterations = 100

t0 = time.perf_counter()
for _ in range(iterations):
    with torch.no_grad():
        _ = ts_model(t_dummy)
latency_ts = (time.perf_counter() - t0) / iterations

t0 = time.perf_counter()
for _ in range(iterations):
    _ = ort_session.run(None, {'input': dummy_input})
latency_onnx = (time.perf_counter() - t0) / iterations

print(f"📊 Binary Model size (TorchScript): {size_ts_mb:.2f} MB")
print(f"📊 Binary Model size (ONNX)       : {size_onnx_mb:.2f} MB")
print(f"📊 Startup loading time (TorchScript): {load_time_ts*1000:.2f} ms")
print(f"📊 Startup loading time (ONNX)       : {load_time_onnx*1000:.2f} ms")
print(f"📊 Inference Latency (TorchScript)   : {latency_ts*1000:.2f} ms")
print(f"📊 Inference Latency (ONNX)          : {latency_onnx*1000:.2f} ms")
print(f"📊 Throughput (TorchScript)          : {1/latency_ts:.1f} FPS")
print(f"📊 Throughput (ONNX)                 : {1/latency_onnx:.1f} FPS")

[INFO] Running benchmark on device: cuda
📊 Binary Model size (TorchScript): 16.25 MB
📊 Binary Model size (ONNX)       : 0.59 MB
📊 Startup loading time (TorchScript): 223.32 ms
📊 Startup loading time (ONNX)       : 139.13 ms
📊 Inference Latency (TorchScript)   : 5.94 ms
📊 Inference Latency (ONNX)          : 4.88 ms
📊 Throughput (TorchScript)          : 168.5 FPS
📊 Throughput (ONNX)                 : 204.8 FPS
